# Flight Data Platform - Ingestion (Bronze Layer)

Este notebook é responsável pela **camada bronze** da plataforma de dados de voos. Ele consome dados de APIs públicas (OpenSky Network e OurAirports), transforma as respostas em Spark DataFrames e persiste tudo em tabelas Delta no catálogo `workspace.default`.

## Fontes de dados ativas

| Fonte | Endpoint | Tabela Delta | Descrição |
| --- | --- | --- | --- |
| OpenSky — States | `/api/states/all` | `bronze_opensky_states` | Estado atual de todas as aeronaves rastreadas (posição, velocidade, altitude, etc.) |
| OurAirports — Airports | `airports.csv` (GitHub raw) | `dim_airports` | Dimensão de aeroportos (IATA, ICAO, coordenadas, país, região) |
| OpenSky — Departures | `/api/flights/departure` | `bronze_opensky_departures` | Voos de partida do aeroporto SBGR (Guarulhos) nas últimas 2 horas |

## Fluxo do pipeline

```
API / CSV  →  Pandas DataFrame  →  Spark DataFrame  →  Tabela Delta (overwrite)
```

# 1. OpenSky - States (`/api/states/all`)

Busca o estado em tempo real de todas as aeronaves rastreadas pelo OpenSky Network. A API pode retornar 17 ou 18 campos por aeronave — o código faz **padding** de valores nulos para garantir um schema consistente de 18 colunas.

In [0]:
import requests
import pandas as pd
import time
from pyspark.sql.functions import current_timestamp, lit

In [0]:
url = "https://opensky-network.org/api/states/all"

response = requests.get(url, timeout=30)
response.raise_for_status()

data = response.json()
states = data.get("states") or []

columns = [
    "icao24",
    "callsign",
    "origin_country",
    "time_position",
    "last_contact",
    "longitude",
    "latitude",
    "baro_altitude",
    "on_ground",
    "velocity",
    "true_track",
    "vertical_rate",
    "sensors",
    "geo_altitude",
    "squawk",
    "spi",
    "position_source",
    "category"
]

# Pad rows that have fewer than 18 fields (category was added in a newer API version)
max_cols = len(columns)
padded_states = [
    list(row) + [None] * (max_cols - len(row))
    if len(row) < max_cols
    else row
    for row in states
]

pdf = pd.DataFrame(padded_states, columns=columns)

df = spark.createDataFrame(pdf)

display(df)

# 2. OurAirports - Dimensão de Aeroportos (airports.csv)

Carrega o dataset público de aeroportos do repositório OurAirports no GitHub. Esta tabela serve como dimensão (dim_airports) para enriquecer os dados de voos com nomes, códigos IATA/ICAO e localização.

In [0]:
airports_url = "https://raw.githubusercontent.com/davidmegginson/ourairports-data/main/airports.csv"
pdf_airports = pd.read_csv(airports_url)

df_airports = spark.createDataFrame(pdf_airports)

display(df_airports.limit(10))

# 3. OpenSky - Partidas (/api/flights/departure)

Consulta voos de partida do aeroporto SBGR (Guarulhos / São Paulo) nas últimas 2 horas. Caso a API retorne erro ou nenhum dado, a célula exibe uma mensagem e df_departures fica como None para que a persistência seja ignorada gracefully.

In [0]:
end_ts = int(time.time())
begin_ts = end_ts - 7200  # last 2 hours
airport = "SBGR"  # Guarulhos (São Paulo)

columns_flights = [
    "icao24",
    "first_seen",
    "est_departure_airport",
    "last_seen",
    "est_arrival_airport",
    "callsign",
    "est_departure_airport_horiz_distance",
    "est_departure_airport_vert_distance",
    "est_arrival_airport_horiz_distance",
    "est_arrival_airport_vert_distance",
    "departure_airport_candidates_count",
    "arrival_airport_candidates_count",
]

url_departures = f"https://opensky-network.org/api/flights/departure?airport={airport}&begin={begin_ts}&end={end_ts}"

try:
    response_dep = requests.get(url_departures, timeout=30)
    response_dep.raise_for_status()
    flights_dep = response_dep.json() or []
except requests.exceptions.HTTPError as e:
    print(f"OpenSky departures API error: {e}")
    flights_dep = []

if flights_dep:
    pdf_departures = pd.DataFrame(flights_dep, columns=columns_flights)
    df_departures = spark.createDataFrame(pdf_departures)
    display(df_departures.limit(10))
else:
    print(f"No departures found for {airport} in the last 2 hours")
    df_departures = None

#4. Persistência em Delta

Grava os DataFrames em tabelas Delta no catálogo workspace.default usando modo overwrite com overwriteSchema=true (permite evolução de schema entre execuções). Cada tabela recebe metadados de ingestão (capture_ts ou ingestion_ts) quando aplicável.


In [0]:
CATALOG = "workspace"
SCHEMA = "default"

# 1. States (OpenSky /states/all)
df_with_ts = df.withColumn("capture_ts", current_timestamp())
df_with_ts.write.option("overwriteSchema", "true").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.bronze_opensky_states")
print(f"bronze_opensky_states: {df_with_ts.count()} rows saved")

# 2. Airports dimension
df_airports.write.option("overwriteSchema", "true").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.dim_airports")
print(f"dim_airports: {df_airports.count()} rows saved")

# 3. Departures (flights/departure)
if df_departures is not None:
    df_departures_meta = df_departures \
        .withColumn("query_airport", lit(airport)) \
        .withColumn("ingestion_ts", current_timestamp())
    df_departures_meta.write.option("overwriteSchema", "true").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.bronze_opensky_departures")
    print(f"bronze_opensky_departures: {df_departures_meta.count()} rows saved")
else:
    print("bronze_opensky_departures: skipped (no data)")

print("\nAll Delta tables saved successfully!")